# Sensitivity 
Author: Ruifei Zhu

July 22, 2024


Calculate the sensitivity of detecting low-frequency exome loci in our studies. Use variants from gnomad v4.1 exome, NFE, 0.0001 < AF < 0.1, passing quality filters as True Positive.

In [1]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

Loading BokehJS ...

/opt/conda/miniconda3/lib/python3.10/site-packages/hail/context.py:352: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 60177
SPARKMONITOR_LISTENER: Application Started: application_1743625867666_0001 ...Start Time: 1743626221943


Running on Apache Spark version 3.3.2
SparkUI available at http://ibd-exome-m.us-central1-a.c.daly-ibd.internal:38811
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.130-bea04d9c79b5
LOGGING: writing to /home/hail/hail-20250402-2036-0.2.130-bea04d9c79b5.log


## Extract gnomad v4.1 nfe <.0001 missense variants 

In [2]:
gnomad_exome_v4_1_sites_ht = hl.read_table('gs://gcp-public-data--gnomad/release/4.1/ht/exomes/gnomad.exomes.v4.1.sites.ht')
gnomad_exome_v4_1_sites_ht.describe()

----------------------------------------
Global fields:
    'freq_meta': array<dict<str, str>> 
    'freq_index_dict': dict<str, int32> 
    'freq_meta_sample_count': array<int32> 
    'faf_meta': array<dict<str, str>> 
    'faf_index_dict': dict<str, int32> 
    'age_distribution': struct {
        bin_edges: array<float64>, 
        bin_freq: array<int32>, 
        n_smaller: int32, 
        n_larger: int32
    } 
    'downsamplings': dict<str, array<int32>> 
    'filtering_model': struct {
        filter_name: str, 
        score_name: str, 
        snv_cutoff: struct {
            bin: int32, 
            min_score: float64
        }, 
        indel_cutoff: struct {
            bin: int32, 
            min_score: float64
        }, 
        snv_training_variables: array<str>, 
        indel_training_variables: array<str>
    } 
    'inbreeding_coeff_cutoff': float64 
    'interval_qc_parameters': struct {
        per_platform: bool, 
        all_platforms: bool, 
        high_qual_

In [3]:
# find index of nfe fre
freq_meta = hl.eval(gnomad_exome_v4_1_sites_ht.freq_meta)
freq_meta

[{'group': 'adj'},
 {'group': 'raw'},
 {'gen_anc': 'afr', 'group': 'adj'},
 {'gen_anc': 'amr', 'group': 'adj'},
 {'gen_anc': 'asj', 'group': 'adj'},
 {'gen_anc': 'eas', 'group': 'adj'},
 {'gen_anc': 'fin', 'group': 'adj'},
 {'gen_anc': 'mid', 'group': 'adj'},
 {'gen_anc': 'nfe', 'group': 'adj'},
 {'gen_anc': 'remaining', 'group': 'adj'},
 {'gen_anc': 'sas', 'group': 'adj'},
 {'group': 'adj', 'sex': 'XX'},
 {'group': 'adj', 'sex': 'XY'},
 {'gen_anc': 'afr', 'group': 'adj', 'sex': 'XX'},
 {'gen_anc': 'afr', 'group': 'adj', 'sex': 'XY'},
 {'gen_anc': 'amr', 'group': 'adj', 'sex': 'XX'},
 {'gen_anc': 'amr', 'group': 'adj', 'sex': 'XY'},
 {'gen_anc': 'asj', 'group': 'adj', 'sex': 'XX'},
 {'gen_anc': 'asj', 'group': 'adj', 'sex': 'XY'},
 {'gen_anc': 'eas', 'group': 'adj', 'sex': 'XX'},
 {'gen_anc': 'eas', 'group': 'adj', 'sex': 'XY'},
 {'gen_anc': 'fin', 'group': 'adj', 'sex': 'XX'},
 {'gen_anc': 'fin', 'group': 'adj', 'sex': 'XY'},
 {'gen_anc': 'mid', 'group': 'adj', 'sex': 'XX'},
 {'gen_an

In [13]:
filtered_ht = gnomad_exome_v4_1_sites_ht.filter(
    hl.any(lambda x: hl.is_defined(x), gnomad_exome_v4_1_sites_ht.vep.transcript_consequences.mane_select)
)

filtered_ht = filtered_ht.select(mane_select=filtered_ht.vep.transcript_consequences.mane_select)

filtered_ht.show()


[Stage 8:=============================>                             (2 + 2) / 4]



,,
locus,alleles,mane_select
locus<GRCh38>,array<str>,array<str>
chr1:62777,"[""A"",""T""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62885,"[""C"",""A""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62901,"[""C"",""A""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62905,"[""G"",""A""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62905,"[""G"",""T""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62911,"[""T"",""C""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62914,"[""A"",""G""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"
chr1:62920,"[""C"",""T""]","[NA,""NM_001005484.2"",NA,""ENST00000641515.2""]"


In [7]:
freq_index_dict = hl.eval(gnomad_exome_v4_1_sites_ht.freq_index_dict)
freq_index_dict

{'XX_adj': 12,
 'XY_adj': 13,
 'adj': 0,
 'afr_XX_adj': 20,
 'afr_XY_adj': 30,
 'afr_adj': 8,
 'ami_XX_adj': 18,
 'ami_XY_adj': 28,
 'ami_adj': 6,
 'amr_XX_adj': 23,
 'amr_XY_adj': 33,
 'amr_adj': 11,
 'asj_XX_adj': 19,
 'asj_XY_adj': 29,
 'asj_adj': 7,
 'eas_XX_adj': 21,
 'eas_XY_adj': 31,
 'eas_adj': 9,
 'fin_XX_adj': 15,
 'fin_XY_adj': 25,
 'fin_adj': 3,
 'hgdp_XX_adj': 164,
 'hgdp_XY_adj': 165,
 'hgdp_adj': 116,
 'hgdp_adygei_XX_adj': 166,
 'hgdp_adygei_XY_adj': 212,
 'hgdp_adygei_adj': 118,
 'hgdp_balochi_XX_adj': 191,
 'hgdp_balochi_XY_adj': 237,
 'hgdp_balochi_adj': 143,
 'hgdp_bantukenya_XX_adj': 169,
 'hgdp_bantukenya_XY_adj': 215,
 'hgdp_bantukenya_adj': 121,
 'hgdp_bantusouthafrica_XX_adj': 171,
 'hgdp_bantusouthafrica_XY_adj': 217,
 'hgdp_bantusouthafrica_adj': 123,
 'hgdp_basque_XX_adj': 172,
 'hgdp_basque_XY_adj': 218,
 'hgdp_basque_adj': 124,
 'hgdp_bedouin_XX_adj': 182,
 'hgdp_bedouin_XY_adj': 228,
 'hgdp_bedouin_adj': 134,
 'hgdp_bergamoitalian_XX_adj': 203,
 'hgdp_ber

In [3]:
GE_nfe_index = gnomad_exome_v4_1_sites_ht.freq_index_dict['nfe_adj'].collect()[0]

In [4]:
gnomad_exome_v4_1_sites_ht.freq[GE_nfe_index].AF.show()

,,
locus,alleles,<expr>
locus<GRCh38>,array<str>,float64
chr1:11994,"[""T"",""C""]",NA
chr1:12016,"[""G"",""A""]",NA
chr1:12060,"[""CTGGAG"",""C""]",0.00e+00
chr1:12074,"[""T"",""C""]",0.00e+00
chr1:12102,"[""G"",""A""]",0.00e+00
chr1:12106,"[""T"",""G""]",0.00e+00
chr1:12138,"[""C"",""A""]",9.80e-03
chr1:12158,"[""C"",""T""]",0.00e+00


In [4]:
# Filter denomitor variants with same filters as numerater variants(passing gnomad v4.1 filters)
gnomad_exome_v4_1_filtered = gnomad_exome_v4_1_sites_ht.filter(
                                                    hl.is_defined(gnomad_exome_v4_1_sites_ht.filters) 
                                                    & (gnomad_exome_v4_1_sites_ht.filters.length() > 0), keep = False)

In [5]:
GGv4_1_ht = hl.read_table("gs://gcp-public-data--gnomad/release/4.1/ht/genomes/gnomad.genomes.v4.1.sites.ht/")

In [6]:
gnomad_exome_v4_1_filtered = gnomad_exome_v4_1_filtered.annotate(gnomad_genomes_v4_1 = GGv4_1_ht[gnomad_exome_v4_1_filtered.key])
gnomad_v4_1_filtered = gnomad_exome_v4_1_filtered.filter(hl.is_defined(gnomad_exome_v4_1_filtered.gnomad_genomes_v4_1.filters) 
                                                               & (gnomad_exome_v4_1_filtered.gnomad_genomes_v4_1.filters.length() > 0), keep = False)

In [9]:
# extract >.0001 nfe missense variants 
gnomad_v4_1_filtered_nfe_lowfreq_missen_mane = gnomad_v4_1_filtered.filter(
    (gnomad_v4_1_filtered.freq[GE_nfe_index].AF >= 0.0001) & 
    (gnomad_v4_1_filtered.freq[GE_nfe_index].AF <= 0.1) &
    (gnomad_v4_1_filtered.vep.most_severe_consequence == 'missense_variant') &
    hl.any(lambda x: hl.is_defined(x), gnomad_v4_1_filtered.vep.transcript_consequences.mane_select) #mane_select
)

In [10]:
gnomad_v4_1_filtered_nfe_lowfreq_missen_mane.freq[GE_nfe_index].AF.export("gs://ibd-exomes-gnomad-subset/QC_round4/7.sensitivity/gnomad_v4_1_nfe_lowfreq_missen_mane.tsv")

2024-08-09 19:55:52.508 Hail: INFO: merging 8790 files totalling 6.7M... / 8789]
2024-08-09 19:56:02.460 Hail: INFO: while writing:
    gs://ibd-exomes-gnomad-subset/QC_round4/7.sensitivity/gnomad_v4_1_nfe_lowfreq_missen_mane.tsv
  merge time: 9.949s
